# Section 4: Toxic comment classification with a Bi-LSTM and GloVe

This notebook trains the Bi-LSTM toxicity classifier on the pooled data from `3_pooled_data.ipynb` (YouTube, Wikipedia, news and Dota chat comments with a binary `label`). It prepares the text, builds a GloVe embedding matrix and tunes the model with a Bayesian search, in which every candidate is scored by k-fold cross-validation on the training split. The search is run twice, once with balanced class weights and once without, so that the two Bi-LSTMs can be compared with the two logistic regressions of notebook 3. The notebook then checks both models on the pooled validation split and saves everything that `4b_LSTM_Logistic_Comparison.ipynb` needs to compare all four models and to run the zero-shot test on r/jobs comments.

Run `3_pooled_data.ipynb` first: it creates `dataset/pooled/pooled_train.csv`, `pooled_val.csv` and `manifest.json`. Sections 2 to 4 below prepare the data and the embeddings, sections 5 and 6 build and tune the models, section 7 shows how the searches went, section 8 evaluates the models, and section 9 lists the files that notebook 4b loads.

## 1. Setup and settings
The notebook first finds the project folder, so that the shared code in `functions/` can be imported and relative paths such as `../dataset` resolve, and then imports the libraries. If the notebook is not inside the project, for example on Colab, it clones the repository instead. A filter hides a harmless warning from the nightly Keras build about LSTM masking; masking itself works.

In [ ]:
import os
import sys
import json
import hashlib
import warnings
import subprocess
from pathlib import Path

# work from this notebook's folder: find the project folder (this one or its parent), or clone it (e.g. on Colab)
here = Path.cwd()
root = next((p for p in (here, here.parent) if (p / 'functions' / 'preprocessing.py').exists()), None)
if root is None:
    root = Path('/content/IS5126_grp6_final_project')
    if not root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/muffinMonday/IS5126_grp6_final_project.git', str(root)], check=True)
assert (root / 'functions' / 'preprocessing.py').exists(), f'Project files are incomplete, please check: {root}'
sys.path.insert(0, str(root))
os.chdir(root / 'notebook')

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from sklearn.metrics import (average_precision_score, precision_score, recall_score, f1_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Embedding
from tensorflow.keras.models import Sequential
from skopt.space import Categorical, Integer, Real

# project code: see functions/__init__.py for what each module contains
from functions.preprocessing import cleanup_text
from functions.embeddings import build_embedding_matrix, download_glove, glove_filename
from functions.tuning import bayes_search

# the nightly Keras build wrongly warns that LSTM / Bidirectional destroy the mask (masking works)
warnings.filterwarnings('ignore', message='.*was passed an input with a mask attached.*')
print('Project folder:', root)

### Settings
Every setting that later cells depend on is defined here, so changing one means editing one cell. The preprocessing settings are reused by the model, and `EMBED_DIM` must match the GloVe file loaded in section 4. The decision threshold is fixed at 0.5: no threshold is tuned, and a comment is flagged as toxic when its predicted probability is at least this value. The two searches share the same search and cross-validation settings, so they differ only in their class weights.

`RETRAIN` decides, for each model, whether section 6 runs a new search or loads the one saved earlier. A search fits about 180 models and is slow, so both switches are off by default: running the whole notebook then never starts a search by accident, and a missing saved result stops with a clear message instead.

In [ ]:
# --- data columns: the pooled text is already lower-cased and cleaned in notebook 3
TEXT_COL = 'text'
LABEL_COL = 'label'  # 1 = toxic, 0 = non-toxic

# --- text preprocessing, reused by the model (EMBED_DIM must match the GloVe file: 50, 100, 200 or 300)
VOCAB_SIZE = 10000
MAX_LENGTH = 80
EMBED_DIM = 100

# --- fixed decision threshold for every prediction
THRESHOLD = 0.5

# --- hyperparameter search: the same for both models, only the class weights differ
SEARCH = {'n_iter': 30, 'n_initial_points': 10, 'objective': 'pr_auc', 'selection': 'stable'}
# n_jobs=-1 trains the 5 folds and the final model in separate processes at the same time; each process needs its
# own memory, so use a smaller number if memory is short, or 1 to train one model after another
CV = {'n_splits': 5, 'epochs': 5, 'batch_size': 64, 'n_jobs': -1}
MODELS = {
    'LSTM (balanced)': {'class_weight': 'balanced', 'save_dir': '../models/bayes_search_pooled_balanced'},
    'LSTM (standard)': {'class_weight': None, 'save_dir': '../models/bayes_search_pooled_unweighted'},
}

# --- run a new search (True) or load the saved one (False), per model
RETRAIN = {'LSTM (balanced)': False, 'LSTM (standard)': False}

## 2. Data
The models learn from the pooled splits written by `3_pooled_data.ipynb`. This section loads them, confirms that they are the splits recorded in the manifest and that their text still matches the current cleaning function, and then looks at how the four sources differ.

### Load the pooled data
The fixed train and validation splits are in `../dataset/pooled/`. Each row has `orig_id`, `source`, `text_raw`, `text` (lower-cased and cleaned) and `label` (1 = toxic), and both splits are stratified by source and label, so they have the same toxic share within every source. The files are checked against `manifest.json`, so the notebook stops if they were edited or regenerated differently. The table shows the number of rows and the toxic share per source and split.

In [ ]:
pooled_dir = '../dataset/pooled'
train_path = os.path.join(pooled_dir, 'pooled_train.csv')
val_path = os.path.join(pooled_dir, 'pooled_val.csv')
manifest_path = os.path.join(pooled_dir, 'manifest.json')
for path in (train_path, val_path, manifest_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} not found. Run notebook/3_pooled_data.ipynb first to create the pooled splits.')

def read_pooled(path):
    # orig_id must stay a string and empty strings must not become NaN
    return pd.read_csv(path, dtype={'orig_id': str}, keep_default_na=False)

train_df, val_df = read_pooled(train_path), read_pooled(val_path)

# the splits must be the ones recorded in the manifest (data, mapping or cleaning may have changed since)
with open(manifest_path, encoding='utf-8') as f:
    manifest = json.load(f)
for name, frame in [('train', train_df), ('val', val_df)]:
    digest = hashlib.sha256(frame.to_csv(index=False, lineterminator='\n').encode()).hexdigest()
    if digest != manifest['split_sha256'][name]:
        raise ValueError(f'pooled_{name}.csv does not match manifest.json: regenerate the splits with notebook 3.')

print(f'Training: {len(train_df):,} comments | validation: {len(val_df):,} comments')
print(f"Toxic share: training {train_df[LABEL_COL].mean():.3f} | validation {val_df[LABEL_COL].mean():.3f}")
summary = (pd.concat([train_df.assign(split='train'), val_df.assign(split='val')])
           .groupby(['source', 'split'])[LABEL_COL].agg(rows='size', toxic_share='mean').unstack('split'))
summary.round(3)

### Check the cleaned text
The `text` column was already produced by `cleanup_text` in notebook 3, so nothing is cleaned again here. To make sure it is still current, a spot check re-cleans a sample of the raw text and stops if the result differs from the stored text, which happens when `cleanup_text` has changed since the splits were made. The same function cleans any new comment at prediction time, so the training and prediction inputs have to match.

In [ ]:
# spot check: the stored text must equal what the current cleanup_text gives for the raw text
check = train_df.sample(min(1000, len(train_df)), random_state=0)
stale = (check['text_raw'].str.lower().apply(cleanup_text) != check[TEXT_COL]).sum()
if stale:
    raise ValueError(f'{stale} of {len(check)} sampled texts differ from the current cleanup_text: re-run notebook 3.')
print(f'Cleaning check passed on {len(check)} sampled training comments.')

### Explore the data
The plots show the number of rows and the toxic share per source, and the distribution of comment lengths in words. The sources differ a lot: Dota chat messages are only a few words long, while news and Wikipedia comments are much longer (notebook 3 has the exact figures). The lengths matter for the next section, because they inform the padded length `MAX_LENGTH`.

In [ ]:
counts = train_df.groupby('source')[LABEL_COL].agg(rows='size', toxic='sum')
lengths = train_df[TEXT_COL].str.split().str.len()

fig, ax = plt.subplots(figsize=(12, 5), nrows=1, ncols=2)

# rows per source, split into non-toxic and toxic
ax[0].bar(counts.index, counts['rows'] - counts['toxic'], color='#4C72B0', label='non-toxic')
ax[0].bar(counts.index, counts['toxic'], bottom=counts['rows'] - counts['toxic'], color='#DD8452', label='toxic')
for i, (rows, toxic) in enumerate(zip(counts['rows'], counts['toxic'])):
    ax[0].text(i, rows + 50, f'{rows}\n({toxic / rows:.0%} toxic)', ha='center', va='bottom', fontsize=10)
ax[0].set_ylabel('Count')
ax[0].set_title('Training rows per source')
ax[0].legend()

# comment length in words per source (outliers hidden)
ax[1].boxplot([lengths[train_df['source'] == s] for s in counts.index], tick_labels=counts.index, showfliers=False)
ax[1].set_ylabel('Comment length (words)')
ax[1].set_title('Comment length per source')
plt.tight_layout()
plt.show()

## 3. Text preprocessing
The network needs fixed-length sequences of integers, so each cleaned comment is turned into word ids and padded to a common length. The tokenizer is fitted on the training text only, so the validation comments don't influence the vocabulary. The steps below run one after the other, and any comment scored later has to go through exactly the same steps, which is why the tokenizer and its settings are saved at the end of this section.

### Tokenize and pad
The tokenizer maps each word to an integer id, keeps the `VOCAB_SIZE` most frequent training words and sends every other word to an unknown-word token. Sequences are then padded or cut at the end to `MAX_LENGTH` tokens, with 0 reserved for padding. The printed shares show how many training comments per source are longer than `MAX_LENGTH` and are therefore truncated.

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')

# fit on the training texts only (this learns the vocabulary)
tokenizer.fit_on_texts(train_df[TEXT_COL])

# convert texts to sequences of integers
train_sequences = tokenizer.texts_to_sequences(train_df[TEXT_COL])
val_sequences = tokenizer.texts_to_sequences(val_df[TEXT_COL])

# example of what this looks like
print('Original:', train_df[TEXT_COL].iloc[0])
print('\nTokenized sequence:', train_sequences[0])

# pad or cut at the end
X_train = pad_sequences(train_sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
X_val = pad_sequences(val_sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
y_train = train_df[LABEL_COL]
y_val = val_df[LABEL_COL]

print('\nPadded shapes:', X_train.shape, X_val.shape)

# share of training comments cut off at MAX_LENGTH, per source
truncated = train_df[TEXT_COL].str.split().str.len().gt(MAX_LENGTH)
print('\nTruncated at MAX_LENGTH:')
print(truncated.groupby(train_df['source']).mean().round(3).to_string())

### Handle empty token sequences
A few comments, such as `! ! ! !`, are not empty but become empty after the tokenizer's filtering. With `mask_zero=True`, a row of only zeros can fail the GPU cuDNN LSTM mask check, so the first padding value of such a row is replaced by the unknown-word id. All rows and labels are kept, the padding stays on the right, and the same rule is applied when new comments are scored. The vocabulary and the pooled files do not change.

In [ ]:
def fix_empty_sequences(X, text_tokenizer=None):
    # one rule for training and prediction: give an all-padding row the unknown-word id in its first position
    text_tokenizer = tokenizer if text_tokenizer is None else text_tokenizer
    X = np.asarray(X).copy()
    if X.ndim != 2 or X.shape[1] == 0:
        raise ValueError('Expected a two-dimensional padded array with positive width.')

    oov_id = text_tokenizer.word_index.get(text_tokenizer.oov_token)
    limit = text_tokenizer.num_words
    if oov_id is None or oov_id <= 0 or (limit is not None and oov_id >= limit):
        raise ValueError('The tokenizer must contain a usable OOV token.')

    empty = (X == 0).all(axis=1)
    X[empty, 0] = oov_id
    return X, int(empty.sum())

X_train, n_train_fixed = fix_empty_sequences(X_train)
X_val, n_val_fixed = fix_empty_sequences(X_val)
print('Empty training sequences repaired:', n_train_fixed)
print('Empty validation sequences repaired:', n_val_fixed)
assert (X_train != 0).any(axis=1).all()
assert (X_val != 0).any(axis=1).all()

### Save the tokenizer
The tokenizer and the padding settings are part of the model: a new comment gets the same input as in training only if it passes through the same tokenizer. They are therefore saved to `../models/tokenizer/`, so that `4b_LSTM_Logistic_Comparison.ipynb` can score other comments, such as the r/jobs sample, without refitting. `settings.json` also records the hash of the training split the tokenizer was fitted on, so notebook 4b can detect a stale tokenizer.

In [ ]:
tokenizer_dir = '../models/tokenizer'
os.makedirs(tokenizer_dir, exist_ok=True)
with open(os.path.join(tokenizer_dir, 'tokenizer.json'), 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
with open(os.path.join(tokenizer_dir, 'settings.json'), 'w', encoding='utf-8') as f:
    json.dump({'vocab_size': VOCAB_SIZE, 'max_length': MAX_LENGTH, 'oov_token': '<OOV>', 'padding': 'post', 'truncating': 'post',
               'train_split_sha256': manifest['split_sha256']['train']}, f, indent=2)
print(f'Saved the tokenizer to {tokenizer_dir}')

## 4. GloVe embeddings
The embedding layer starts from pre-trained GloVe vectors instead of random ones, so words with similar meanings begin with similar representations. The matrix has one row per tokenizer id: row *i* holds the GloVe vector of the word with id *i*. The code uses the local copy in `../glove` and downloads the file there first if it is missing, and words that GloVe doesn't know get small random vectors. The printed coverage is the share of distinct vocabulary words found in GloVe.

In [ ]:
# load pre-trained GloVe vectors, preferring the local copy in ../glove
glove_dir = '../glove'
glove_path = os.path.join(glove_dir, glove_filename(EMBED_DIM))

try:
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, VOCAB_SIZE)
except FileNotFoundError:
    print(f'{glove_path} not found.')
    download_glove(EMBED_DIM, glove_dir)
    embedding_matrix, glove_coverage = build_embedding_matrix(glove_path, tokenizer.word_index, VOCAB_SIZE)

print('Embedding matrix shape:', embedding_matrix.shape)
print(f'Vocabulary found in GloVe: {glove_coverage:.1%}')

## 5. Model and search space
`build_model` assembles the network from four hyperparameters: a GloVe-initialised embedding layer that is either frozen or fine-tuned and masks the padding, a bidirectional LSTM that reads each comment forwards and backwards, and a sigmoid output that gives the probability of toxic. The search space states which values each hyperparameter may take: whether the embedding is trainable, the LSTM size, the dropout rate and the learning rate. The size and the learning rate are searched on a log scale, so small values get as much coverage as large ones. The number of epochs is fixed in the settings to keep each trial affordable.

In [ ]:
def build_model(trainable, lstm_units, dropout, lr):
    m = Sequential([
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBED_DIM,
            embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
            trainable=trainable,
            mask_zero=True,  # LSTM skips the padding (index 0)
        ),
        Bidirectional(LSTM(lstm_units, dropout=dropout)),
        Dense(1, activation='sigmoid'),
    ])
    m.compile(
        loss='binary_crossentropy',
        optimizer=tf.keras.optimizers.Adam(lr),
        metrics=[
            'accuracy',
            tf.keras.metrics.F1Score(name='f1_score', threshold=THRESHOLD),
            tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
        ],
    )
    return m

search_space = {
    'trainable': Categorical([True, False]),  # fine-tune GloVe or keep it frozen
    'lstm_units': Integer(8, 128, prior='log-uniform'),
    'dropout': Real(0.0, 0.5),
    'lr': Real(3e-4, 1e-2, prior='log-uniform'),
}

## 6. Hyperparameter search
A Bayesian search proposes a set of hyperparameters, scores it with stratified 5-fold cross-validation on the training split, and uses the score to choose the next set to try. The score is the mean validation PR-AUC across the folds. The top trials are usually within noise of each other, so instead of taking the single best mean, the search picks, among the trials within one fold standard deviation of the best, the one with the lowest standard deviation across folds (`selection='stable'`). Its final model is trained on the full training split and scored on the validation split.

The search is run twice with identical settings and the same random seed, changing only the class weights. Toxic comments are about 15% of the training split, so the balanced run weights the training loss by class: each class gets the weight n / (2 × its count), computed from the labels each model trains on, which is about 0.59 for non-toxic and 3.3 for toxic comments and is the weighting of the balanced logistic regression in notebook 3. The standard run trains on the original class proportions. The weights change training only, and every metric is still computed from the predicted probabilities. Balanced weights push the probabilities up, so at the fixed 0.5 cutoff the balanced model is expected to flag more toxic comments, with higher recall and lower precision.

A finished search is saved to its own folder and loaded the next time, unless `RETRAIN` says otherwise. A loaded result is not checked against the data, the preprocessing, the class weights or the search space, so set `RETRAIN` to True for a model after changing any of them.

### Train or load a search
`run_or_load` returns the search and its best model for one of the two configurations. With `RETRAIN` on, it runs the search and saves it; otherwise it loads the saved result, and stops with a clear message if there is none or if the saved model was trained with other class weights. Either way the fixed threshold is then applied to the model, which also replaces any cutoff stored in an older saved result, and a short summary of the selected trial is printed.

In [ ]:
def run_or_load(name):
    config = MODELS[name]
    save_dir = config['save_dir']
    saved = os.path.exists(os.path.join(save_dir, 'search_state.pkl'))

    if RETRAIN[name]:
        search = bayes_search(
            build_model, search_space, X_train, y_train, X_val, y_val, **SEARCH,
            kfold_kwargs={**CV, 'class_weight': config['class_weight']},
        )
        best = search.fit()
    elif saved:
        search = bayes_search.load(save_dir)
        best = search.best_kfold
        # results saved before the option existed had no class weights
        trained_with = getattr(best, 'class_weight', None)
        if trained_with != config['class_weight']:
            raise ValueError(f'{name}: the result in {save_dir} was trained with class_weight={trained_with!r}, expected {config["class_weight"]!r}.')
        print(f'Loaded {name} from {save_dir}')
    else:
        raise FileNotFoundError(f"No saved search for {name} in {save_dir}. Set RETRAIN['{name}'] = True to train it.")

    # the fixed threshold applies to every model, and replaces any cutoff stored in an older saved result
    best.threshold = THRESHOLD
    best.tuned_threshold = None
    best.tuned_validation_metrics = None

    if RETRAIN[name]:
        search.save(save_dir)
        print(f'Saved {name} to {save_dir}')

    print(f'{name}: selected trial', getattr(search, 'best_trial', None), '| params:', search.best_params)
    print(f'{name}: hold-out @ threshold {THRESHOLD:g}:', {k: round(v, 4) for k, v in best.validation_metrics.items()})
    return search, best

searches, models = {}, {}

### Balanced model
Runs or loads the search with balanced class weights (`class_weight='balanced'`). The saved result is in `../models/bayes_search_pooled_balanced`. The table lists the five trials with the best mean PR-AUC.

In [ ]:
searches['LSTM (balanced)'], models['LSTM (balanced)'] = run_or_load('LSTM (balanced)')
searches['LSTM (balanced)'].results.sort_values('avg_val_pr_auc', ascending=False).head()

### Standard model
Runs or loads the same search without class weights (`class_weight=None`). The saved result is in `../models/bayes_search_pooled_unweighted`. The table again lists the five best trials.

In [ ]:
searches['LSTM (standard)'], models['LSTM (standard)'] = run_or_load('LSTM (standard)')
searches['LSTM (standard)'].results.sort_values('avg_val_pr_auc', ascending=False).head()

### Collect the models
From here on the two models are handled together. `models` and `searches` hold them under the names used in notebook 4b, and the validation split is scored once per model, so that every later cell uses the same predictions at the same threshold.

In [ ]:
model_colors = {'LSTM (balanced)': '#4C72B0', 'LSTM (standard)': '#55A868'}

# validation probabilities and predictions at each model's threshold
val_proba_by_model = {name: kf.final_model.predict(X_val, verbose=0).ravel() for name, kf in models.items()}
val_pred_by_model = {name: (val_proba_by_model[name] >= kf.threshold).astype(int) for name, kf in models.items()}

## 7. Search diagnostics
The plots show the validation PR-AUC (mean ± standard deviation across the CV folds) for every round of each search, the balanced model on the left and the standard one on the right, on the same y-axis. The dotted line marks the end of the random initial points and the orange dot is the selected trial. They help to judge whether each search was still improving when it stopped, and how large the fold-to-fold noise is compared with the differences between rounds, which is why near-ties are resolved by stability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (name, s) in zip(axes, searches.items()):
    res = s.results
    mean, std = res['avg_val_pr_auc'], res['std_val_pr_auc']
    ax.errorbar(res.index, mean, yerr=std, fmt='o-', color=model_colors[name], ecolor=to_rgba(model_colors[name], 0.4),
                capsize=3, label='mean ± std across folds')
    ax.axvline(s.n_initial_points + 0.5, color='grey', linestyle=':', label='end of random initial points')
    selected_trial = getattr(s, 'best_trial', None) or mean.idxmax()
    ax.scatter([selected_trial], [mean[selected_trial]], color='#DD8452', zorder=3, s=80, label=f'selected (trial {selected_trial})')
    ax.set_xlabel('Search round')
    ax.set_title(f'{name}: PR-AUC per round of Bayesian search')
    ax.legend()
axes[0].set_ylabel('Validation PR-AUC')
plt.tight_layout()
plt.show()

## 8. Validation performance
The selected models are now scored on the held-out validation split, which was never used for training or tuning. It contains comments from the same four sources as the training data, so these scores are in-domain and optimistic about r/jobs; the zero-shot test on r/jobs is in `4b_LSTM_Logistic_Comparison.ipynb`. The three parts below look at the same predictions from different angles: the metrics and confusion matrices at the fixed threshold, the precision–recall curves across all thresholds, and a table of what other cutoffs would do.

### Metrics and confusion matrices
The table reports the proposal's metrics for each model at the fixed 0.5 threshold. Precision and recall are for the toxic class, and macro F1 averages both classes. PR-AUC is the average precision of the predicted probabilities, so it does not depend on the threshold, and it matches `kfold_wrapper`. Each confusion matrix shows how many toxic comments the model catches and how many clean ones it flags.

In [ ]:
actual = np.asarray(y_val).astype(int)

# precision, recall and macro F1 at each model's threshold; PR-AUC (average precision) from the probabilities
val_metrics = pd.DataFrame([{
    'n_comments': len(actual),
    'threshold': models[name].threshold,
    'precision': precision_score(actual, pred, zero_division=0),
    'recall': recall_score(actual, pred, zero_division=0),
    'macro_f1': f1_score(actual, pred, average='macro', zero_division=0),
    'pr_auc': average_precision_score(actual, val_proba_by_model[name]),
} for name, pred in val_pred_by_model.items()], index=list(models))
display(val_metrics.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, pred) in zip(axes, val_pred_by_model.items()):
    ConfusionMatrixDisplay.from_predictions(
        actual, pred, display_labels=['Non-toxic', 'Toxic'], cmap='Blues', colorbar=False, ax=ax,
    )
    ax.set_title(f'{name}: validation confusion matrix')
plt.tight_layout()
plt.show()

### Precision–recall curves
Precision (the share of flagged comments that are toxic) is plotted against recall (the share of toxic comments caught) as the cutoff moves from 1 to 0, for both models on the same axes. The area under each curve is the PR-AUC, which summarises how well a model ranks toxic above clean comments whatever the cutoff, and it is the metric the searches maximise. The dashed line is the toxic share of the validation split, which is the PR-AUC of random ranking, and the dot on each curve marks that model's operating point at the fixed threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, kf in models.items():
    proba, pred = val_proba_by_model[name], val_pred_by_model[name]
    precision, recall, _ = precision_recall_curve(actual, proba)
    ax.step(recall, precision, where='post', color=model_colors[name], linewidth=2,
            label=f'{name} (PR-AUC = {average_precision_score(actual, proba):.3f})')
    # operating point at the model's threshold
    op_recall = ((pred == 1) & (actual == 1)).sum() / (actual == 1).sum()
    op_precision = ((pred == 1) & (actual == 1)).sum() / max((pred == 1).sum(), 1)
    ax.scatter([op_recall], [op_precision], color=model_colors[name], edgecolors='black', zorder=3, s=70,
               label=f'{name} threshold = {kf.threshold:g}')
ax.axhline(actual.mean(), color='grey', linestyle='--', label=f'Chance (PR-AUC = {actual.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.set_title('Validation precision-recall curves')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

### Cutoff trade-off
The fixed cutoff is a choice, so these tables show what other cutoffs would do. For each model they give the recall, the precision, the false positive rate and the share of comments flagged at several cutoffs, on the out-of-fold predictions of the training split (`oof_*`) and, as a sanity check, on the validation split (`val_*`). Lowering the cutoff catches more toxic comments but flags more clean ones, so the tables help to judge whether the extra recall is worth the extra moderation work. They are for reference and do not change the threshold. Training and validation have the same toxic share, so precision here is in-domain; on r/jobs the share of toxic comments is unknown and may be lower, and precision would then be lower too.

In [ ]:
# recall = share of toxic comments caught | precision = share of flagged comments that are toxic
# fpr = share of non-toxic comments flagged | flagged = share of all comments flagged
for name, kf in models.items():
    print(name)
    display(kf.threshold_table().round(2))

## 9. Saved outputs
Notebook 4b loads the files below, so it needs no training. Each path is relative to this notebook.

| Path | Saved by | Contents |
|---|---|---|
| `../models/bayes_search_pooled_balanced/` | section 6, balanced model | the search results and `best_kfold/` with the final Keras model |
| `../models/bayes_search_pooled_unweighted/` | section 6, standard model | the same for the model without class weights |
| `../models/tokenizer/` | section 3 | `tokenizer.json` and `settings.json` with the padding settings |

The two logistic regressions that notebook 4b compares them with are saved by `3_pooled_data.ipynb` in `../models/baselines/`.